In [1]:
from discovery_child_development import PROJECT_DIR
import pandas as pd
import json

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'
PATH_TO_DATASET = ENRICHED_DATA_DIR / 'openalex_relevance_labels_only_relevant.csv'

# Path to topic information
PATH_TO_TOPICS = PROJECT_DIR / "discovery_child_development/pipeline/labelling/taxonomy_cat/prompts/topics.json"

# Load topic information
topics_dict = json.load(open(PATH_TO_TOPICS, 'r'))
topics = list(topics_dict.keys())
print(len(topics))

38


In [2]:
# Load the data with texts
text_df = (
    pd.read_csv(PATH_TO_DATASET)
    .assign(id = lambda df: df.id.apply(lambda x: x.split("/")[-1]))
)
len(text_df)

37975

In [4]:
dfs = []
for topic in topics:
    keywords = topics_dict[topic]["filtering_keywords"]
    df = (
        pd.read_csv(ENRICHED_DATA_DIR / f'taxonomy_cat/openalex/taxonomy_cat_predictions_{topic}.csv')
        .merge(text_df[['id', 'text']], on='id', how='left')
    )
    keyword_hits = (
        df.text
        .str.lower()
        .str.replace(r'[^a-zA-Z0-9]', ' ', regex=True)
        .str.contains("|".join(keywords))
    )
    df = df[keyword_hits]
    dfs.append(df)

In [5]:
labelled_df = (
    pd.concat(dfs, ignore_index=True)
    # Key step: taking only data that's robustly relevant
    .query("prediction==1.0")
    .groupby("id")
    .agg(topics = ("topic", list))
    .reset_index()
)

In [6]:
labelled_text_df = (
    text_df
    .merge(labelled_df, on='id', how='left')
    .set_index("id")
)

In [7]:
# Double check specific topics
extra_keywords = {
    "ai2": ["artificial intelligence", "data science", "machine learning", "deep learning", "chatbot", "natural language processing", "computer vision", "convolutional neural network", "recurrent neural network", "reinforcement learning", "predictive model", "predictive analytics"],
    "ar_vr": ["virtual reality", "augmented reality", "mixed reality"],
    "social_media": ["social media"],
    "robotics": ["robot"],
    "parenting2": ["home learning environment", "home learning", "parenting approach", "parenting style", "home learning",
    "parenting style",
    "single parent",
    "parenting skill",
    "parenting education",
    "parenting program",
    "parenting intervention",
    "parenting support",
    "parenting practice",
    "parenting behavior",
    "parenting knowledge",
    "parenting attitude",
    "parenting guidance",
    "parenting stress",
    "parent skill",
    "parent education",
    "parent program",
    "parent intervention",
    "parent support",
    "parent practice",
    "parent behavior",
    "parent knowledge",
    "parent attitude",
    "parent guidance",
    "parent stress"],
    "wearables": ["wearable", "internet of things", " iot "],
    "mobile": ["smartphone", 'ipad', 'iphone', 'android', 'phone application'],
    "infancy": ["infant", "newborn", "neonate"],
    "protection": ["child protection", "safeguarding"],
    "communication": ["language development", "speech development"],
    "cognitive": ["cognitive development"],
    "send": ["autism", "adhd", "learning disability", "special educational needs"],
    "mental_health": [" mental health "],
    "rct": ["randomised control trial", "randomized control trial"],
    "social_services": ["social service"],
    "mobile": ['mobile phone', 'smartphone', 'android', 'iphone'],
}
extra_keywords_patents = {
    "preschool": ['preschool']
}


In [8]:
for topic in extra_keywords:
    keywords = extra_keywords[topic]
    keywords = [word.lower() for word in keywords]
    hits_df = text_df[(text_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ', regex=True).str.contains("|".join(keywords)) == True)]
    hits_ids = hits_df.id.to_list()
    labelled_text_df.loc[hits_ids, "topics"] = labelled_text_df.loc[hits_ids, "topics"].apply(lambda x: list(set(x + [topic])) if isinstance(x, list) else [topic])

In [8]:
# for topic in extra_keywords_patents:
#     keywords = extra_keywords_patents[topic]
#     keywords = [word.lower() for word in keywords]
#     hits_df = text_df[(text_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ', regex=True).str.contains("|".join(keywords)) == True)]
#     hits_ids = hits_df.query("source == 'patents'").id.to_list()
#     labelled_text_df.loc[hits_ids, "topics"] = labelled_text_df.loc[hits_ids, "topics"].apply(lambda x: list(set(x + [topic])) if isinstance(x, list) else [topic])

In [11]:
text_labelled_df = (
    pd.read_csv(PATH_TO_DATASET)
    .assign(id = lambda df: df.id.apply(lambda x: x.split("/")[-1]))
    .merge(labelled_text_df.reset_index()[['id', 'topics']], on="id", how="left")
    .assign(topics = lambda df: df.topics.apply(lambda x: ", ".join(x) if type(x) == list else x))
    .drop(columns=["predictions"])
)

In [12]:
# Papers with no labels
n_with_topics = (text_labelled_df.topics.isnull() == False).sum()
n_without_topics = text_labelled_df.topics.isnull().sum()
n_with_topics / len(text_labelled_df), n_without_topics / len(text_labelled_df)

(0.9284265964450297, 0.07157340355497037)

In [13]:
print(n_with_topics)

35257


In [14]:
text_labelled_df.to_csv(ENRICHED_DATA_DIR / 'taxonomy_cat/taxonomy_cat_predictions_openalex_filtered.csv', index=False)

## Combine with the concept-derived data

In [33]:
text_labelled_old_df = (
    pd.read_csv(
        ENRICHED_DATA_DIR / "taxonomy_cat/taxonomy_cat_predictions_filtered.csv"
    )
    .assign(id=lambda df: df["id"].apply(lambda x: x.split("/")[-1]))
    .query("source == 'openalex'")
    # .rename(columns={"source": "dataset"})
    )

In [34]:
keyword_ids = text_labelled_df.id.to_list()

In [37]:
text_labelled_df_final = (
    pd.concat([text_labelled_df, text_labelled_old_df.query("id not in @keyword_ids")], ignore_index=True)
    .assign(source='openalex')
    .drop_duplicates(subset=['id'])
)

In [38]:
text_labelled_df_final

,id,text,source,topics
0,W2102925803,Interventions for increasing fruit and vegetab...,openalex,"rct, nutrition, health"
1,W3097547541,Does Household Income Affect children’s Outcom...,openalex,"mental_health, income, rct, health"
2,W3033628182,Global burden of preterm birth. Abstract Prete...,openalex,"prenatal, infancy"
3,W3106925869,Screening tools for early identification of ch...,openalex,"nontech, communication, send, infancy"
4,W4234954338,Strategies to improve the implementation of he...,openalex,"rct, policy, nutrition, health"
...,...,...,...,...
70918,W4385782293,Virtual assessment of stress reactivity in you...,openalex,"operations, mental_health"
70919,W4365815550,Infant-Directed Communication: Examining the m...,openalex,"infancy, communication"
70920,W4385765606,It's not just what we don't know: The mapping ...,openalex,communication
70921,W4379911316,Prenatal diagnosis of Persistent Left Superior...,openalex,infancy


In [41]:
text_labelled_df_final.to_csv(ENRICHED_DATA_DIR / 'taxonomy_cat/taxonomy_cat_predictions_openalex_filtered_final.csv', index=False)

## Metadata

In [58]:
metadata_df = []
for i in range(0,7):
    df = pd.read_parquet(PROJECT_DIR / f'inputs/data/openalex_keywords/parquet/output_{i}.parquet')
    pub_countries = []
    for i, row in df.iterrows():
        pub_country = []
        for author in row['authorships']:
            for institution in author['institutions']:
                pub_country.append(institution['country_code'])
        pub_countries.append(set(pub_country))
    ids = [pub.split("/")[-1] for pub in df.id.to_list()]
    metadata_df.append(
        pd.DataFrame({'id': ids, 'country_code': pub_countries, "year": df.publication_year})
        .assign(country_code = lambda x: x['country_code'].apply(lambda y: list(y)))
    )

In [59]:
metadata_df = pd.concat(metadata_df, ignore_index=True)

In [61]:
metadata_df.drop_duplicates(subset=['id'], inplace=True)

In [81]:
metadata_df.head(
)

,id,country_code,year
0,W3038513886,[CA],2020
1,W3006659024,[GB],2020
2,W3014541161,[US],2020
3,W2102925803,[AU],2020
4,W3026186606,[GB],2020


In [89]:
metadata_df[metadata_df.country_code.isnull()].groupby('year').size()

Series([], dtype: int64)

In [105]:
import ast
old_country_codes = (
    pd.read_csv(ENRICHED_DATA_DIR / "pubs_metadata_df.csv")
    .assign(country_code=lambda df: df["country_code"].apply(lambda x: ast.literal_eval(x)))
)

In [97]:
# old_country_codes[old_country_codes.country_code.isnull()]

In [106]:
old_metadata = (
    pd.read_csv(ENRICHED_DATA_DIR / "openalex_concepts_metadata.csv")
    .assign(id = lambda df: df.openalex_id.apply(lambda x: x.split("/")[-1]))
    .rename(columns={"publication_year": "year"})
    .merge(
        old_country_codes,
        on="id",
        how="left"
    )
)[['id', 'year', 'country_code']]


In [107]:
metadata_final_df = (
    pd.concat([metadata_df[['id', 'year', 'country_code']], old_metadata[['id', 'year', 'country_code']]], ignore_index=True)
    .drop_duplicates(subset=['id'])
    .assign(country_code = lambda df: df.country_code.apply(lambda x: str(x)))
)

In [109]:
metadata_final_df.to_csv(ENRICHED_DATA_DIR / 'openalex_metadata_df_final.csv', index=False)

In [102]:
import ast 

data_df = (
    pd.read_csv(
        ENRICHED_DATA_DIR / "taxonomy_cat/taxonomy_cat_predictions_openalex_filtered_final.csv"
    )
    .assign(id=lambda df: df["id"].apply(lambda x: x.split("/")[-1]))
    .query("source == 'openalex'")
    .rename(columns={"source": "dataset"})
)

In [115]:
len(data_df)

37975

In [111]:
metadata_df = (
    pd.read_csv(ENRICHED_DATA_DIR / 'openalex_metadata_df_final.csv')
    .drop_duplicates(subset=["id"])
    .dropna(subset=["country_code"])
    .assign(country_code=lambda df: df["country_code"].apply(lambda x: ast.literal_eval(x)))
)

In [112]:
df = (
    data_df.merge(metadata_df[["id", "year", "country_code"]], how="left", on="id")
    .query("year >= 2013 and year <= 2023")
)

In [114]:
df.country_code.isnull().sum()


0